# MILESTONE - 3

In [ ]:
# ============================================================
# MILESTONE 3 - RAG SETUP (given, with the small typo fixed)
# ============================================================

!pip install faiss-cpu

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")


# ============================================================
# Zero-shot classifier setup (given)
# ============================================================

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])


# ============================================================
# Q1 - zero-shot probability of the correct option, row 150
# ============================================================

result_q1 = zs(prompt_150, candidate_labels=labels_150)

# result_q1['labels'] and result_q1['scores'] are sorted by score, highest first.
# Find the score that matches the correct answer text (ans_150).
q1_score = None
for label, score in zip(result_q1['labels'], result_q1['scores']):
    if label == ans_150:
        q1_score = score

print("Q1 answer:", round(q1_score, 3))


# ============================================================
# Q2 - FAISS rank of the true document for row 150
# ============================================================

prompt_150_embedding = model.encode([prompt_150])
distances, retrieved_indices = index.search(prompt_150_embedding, 10)

# retrieved_indices[0] is a list of the top 10 KB indices, in order (rank 1 first)
q2_rank = None
for rank, kb_index in enumerate(retrieved_indices[0]):
    if kb_index == 150:
        q2_rank = rank + 1  # rank is 1-based

print("Q2 answer:", q2_rank)


# ============================================================
# Q3 - Cross-Encoder rerank: new rank of the true document
# ============================================================

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices[0]]
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Sort the 10 documents by cross-encoder score, highest first.
# We keep track of the ORIGINAL kb index for each document so we can find rank 150.
combined = list(zip(retrieved_indices[0], ce_scores))
combined_sorted = sorted(combined, key=lambda x: x[1], reverse=True)

q3_rank = None
for rank, (kb_index, score) in enumerate(combined_sorted):
    if kb_index == 150:
        q3_rank = rank + 1

print("Q3 answer:", q3_rank)


# ============================================================
# Q4 - token count for row 42's RAG string (top 5 docs)
# ============================================================

prompt_42 = str(train.iloc[42]['prompt'])
prompt_42_embedding = model.encode([prompt_42])
distances_42, retrieved_indices_42 = index.search(prompt_42_embedding, 5)

docs_42 = [kb[i] for i in retrieved_indices_42[0]]
concatenated_docs_42 = " ".join(docs_42)

rag_string_42 = "Context: " + concatenated_docs_42 + " Question: " + prompt_42

bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokens_42 = bert_tokenizer(rag_string_42, truncation=False)

q4_token_count = len(tokens_42['input_ids'])
print("Q4 answer:", q4_token_count)


# ============================================================
# Q5 - zero-shot with TRUE document as context, row 150
# ============================================================

true_document_150 = kb[150]
rag_string_150_true = "Context: " + true_document_150 + " Question: " + prompt_150

result_q5 = zs(rag_string_150_true, candidate_labels=labels_150)

q5_score = None
for label, score in zip(result_q5['labels'], result_q5['scores']):
    if label == ans_150:
        q5_score = score

print("Q5 answer:", round(q5_score, 3))


# ============================================================
# Q6 - Adversarial RAG: wrong document (kb index 999), row 150
# ============================================================

wrong_document_999 = kb[999]
rag_string_150_adversarial = "Context: " + wrong_document_999 + " Question: " + prompt_150

result_q6 = zs(rag_string_150_adversarial, candidate_labels=labels_150)

q6_score = None
for label, score in zip(result_q6['labels'], result_q6['scores']):
    if label == ans_150:
        q6_score = score

print("Q6 answer:", round(q6_score, 3))


# ============================================================
# Q7 - Hit Rate for rows 0-99 (top k=5 retrieval)
# ============================================================

hit_count = 0

for i in range(100):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_text = str(row[row['answer']])

    prompt_i_embedding = model.encode([prompt_i])
    dist_i, idx_i = index.search(prompt_i_embedding, 5)
    retrieved_docs_i = [kb[j] for j in idx_i[0]]

    is_hit = False
    for doc in retrieved_docs_i:
        if correct_text in doc:
            is_hit = True

    if is_hit:
        hit_count = hit_count + 1

hit_rate = (hit_count / 100) * 100
print("Q7 answer:", round(hit_rate, 1))


# ============================================================
# Q8 - Full RAG pipeline (retrieve -> rerank -> augment ->
#      predict -> score) for rows 0-19, average MAP@3
# ============================================================

OPTIONS = ['A', 'B', 'C', 'D', 'E']

def average_precision_at_3(correct_letter, ranked_letters):
    # ranked_letters is a list of 3 letters, best guess first
    score = 0.0
    hits = 0
    for position, letter in enumerate(ranked_letters):
        if letter == correct_letter:
            hits = hits + 1
            score = score + (hits / (position + 1))
    return score

ap_scores = []

for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_letter_i = row['answer']

    # Step 1: Retrieve top 5
    prompt_i_embedding = model.encode([prompt_i])
    dist_i, idx_i = index.search(prompt_i_embedding, 5)
    retrieved_docs_i = [kb[j] for j in idx_i[0]]

    # Step 2: Rerank with Cross-Encoder, pick the single best document
    pairs_i = [[prompt_i, doc] for doc in retrieved_docs_i]
    scores_i = cross_encoder.predict(pairs_i)
    best_doc_index = int(np.argmax(scores_i))
    best_doc = retrieved_docs_i[best_doc_index]

    # Step 3: Augment
    rag_string_i = "Context: " + best_doc + " Question: " + prompt_i

    # Step 4: Predict
    option_texts_i = [str(row[o]) for o in OPTIONS]
    result_i = zs(rag_string_i, candidate_labels=option_texts_i)

    # Step 5: Score - build a letter -> score map, then sort by score
    letter_scores = {}
    for o in OPTIONS:
        option_text = str(row[o])
        for label, score in zip(result_i['labels'], result_i['scores']):
            if label == option_text:
                letter_scores[o] = score

    # sort letters by score, highest first, take top 3
    sorted_letters = sorted(letter_scores, key=lambda letter: letter_scores[letter], reverse=True)
    top3_letters = sorted_letters[:3]

    ap = average_precision_at_3(correct_letter_i, top3_letters)
    ap_scores.append(ap)

q8_map_at_3 = sum(ap_scores) / len(ap_scores)
print("Q8 answer:", round(q8_map_at_3, 3))

Creating knowledge base
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q4 answer: 216
Q5 answer: 0.989
Q6 answer: 0.529
Q7 answer: 73.0


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q8 answer: 0.975
